# C8-embeddings — Practice p13 — Solution

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

QUERY = "lamp"
CANDIDATES = ["candle", "beacon", "quokkas", "flashlight", "chandelier"]

present = [word for word in CANDIDATES if word in kv.key_to_index]
n_missing = int(len(CANDIDATES) - len(present))

Vc = np.asarray(kv[present], dtype=np.float64)
Wc = Vc / np.sqrt((Vc * Vc).sum(axis=1, keepdims=True))
q = np.asarray(kv[QUERY], dtype=np.float64)
q = q / np.sqrt((q * q).sum())

sims = Wc @ q
order = np.argsort(sims)[::-1]
# The query is absent from the candidates, so there is no self row to exclude.
ranked = np.asarray(present)[order].tolist()
best = str(ranked[0])

gensim_sims = np.asarray([kv.similarity(QUERY, word) for word in present],
                         dtype=np.float64)
max_gap = float(np.max(np.abs(sims - gensim_sims)))

The manual ranking is `candle`, `flashlight`, `beacon`, `chandelier`. The cross-check uses `1e-5` because the library computes from its native float32 artifact, while the manual pipeline begins by converting vectors to float64; `1e-12` would demand agreement beyond the cross-register rounding precision.

### Answer check

In [ ]:
assert present == ["candle", "beacon", "flashlight", "chandelier"]
assert n_missing == 1
assert Wc.shape == (4, 100) and Wc.dtype == np.float64
assert q.shape == (100,) and q.dtype == np.float64
assert np.allclose(sims,
                   np.array([0.6780828130916523, 0.5311179085250453,
                             0.5869534077294049, 0.45033821388858675]),
                   atol=1e-12, rtol=0)
assert ranked == ["candle", "flashlight", "beacon", "chandelier"]
assert best == "candle"
assert max_gap < 1e-5